# Show Your Work — LLM Extraction Prototype
Take messy natural-language financial input → structured JSON with labeled assumptions + clarification questions.




In [6]:
!pip install -q openai


In [7]:
from openai import OpenAI
import json

OPENROUTER_API_KEY = ""

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL = "google/gemini-2.5-flash"



print(f"Client ready — using {MODEL} via OpenRouter")


✅ Client ready — using google/gemini-2.5-flash via OpenRouter


## System Prompt
This is the core of your work. It tells the LLM *how* to extract, *what* to label, and *when* to ask questions. Tune this until all 3 test cases look right.


In [8]:
SYSTEM_PROMPT = """\
You are a financial-assumption extractor for a "Can I afford it?" decision tool.

Your job:
1. Read the user's plain-language description of their financial situation and goal.
2. Extract every relevant financial variable into structured JSON.
3. Label each variable's source:
   - "stated"   → the user explicitly said this
   - "inferred" → you calculated or reasoned this from what they said
   - "missing"  → you need this but the user didn't mention it
4. For "missing" or uncertain variables, write a SHORT, friendly clarification question.
5. Do NOT give financial advice. Do NOT run calculations. Just extract and label.

Return ONLY valid JSON matching this exact schema (no markdown, no extra text):

{
  "assumptions": [
    {
      "variable": "monthly_income",
      "value": <number or null if missing>,
      "unit": "$/month",
      "source": "stated" | "inferred" | "missing",
      "source_detail": "Brief explanation of where this came from or why it's missing"
    }
  ],
  "clarification_questions": [
    {
      "target_variable": "variable_name",
      "question": "The friendly question to ask the user",
      "why_it_matters": "One sentence on how this affects the outcome"
    }
  ],
  "goal_summary": "One-sentence restatement of what the user wants to figure out",
  "confidence_note": "One sentence about overall confidence in the extraction"
}

Guidelines for the assumptions array — include ALL of these variables, marking as
"missing" if not mentioned:
  - monthly_income
  - monthly_expenses (total, or broken into categories if user gave them)
  - monthly_surplus (income minus expenses — inferred if both are stated)
  - current_savings
  - goal_amount
  - goal_deadline (in months from now, or a date)
  - goal_description

If the user gives expense categories (rent, food, etc.), include each as a separate
assumption AND a total. If they give a range ("$700-800"), use the midpoint and note
the range in source_detail.

Be conservative: if something is ambiguous, mark it "inferred" and add a
clarification question. Never silently assume critical values.
"""

print(f"System prompt loaded ({len(SYSTEM_PROMPT)} chars)")


System prompt loaded (2096 chars)


## Helper Functions

In [9]:
def call_llm(user_message: str) -> dict:
    """Send user input to the LLM via OpenRouter, get back structured JSON."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.2,           # low temp for consistent extraction
        response_format={"type": "json_object"},  # force JSON output
    )

    raw_text = response.choices[0].message.content

    # Parse JSON (strip markdown fences if model adds them)
    cleaned = raw_text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1]
        cleaned = cleaned.rsplit("```", 1)[0]

    return json.loads(cleaned)


def print_extraction(result: dict):
    """Pretty-print an extraction result."""
    print(f"\n  Goal: {result.get('goal_summary', 'N/A')}")
    print(f"  Confidence: {result.get('confidence_note', 'N/A')}")

    print("\n--- ASSUMPTIONS ---")
    for a in result.get("assumptions", []):
        val = a.get("value")
        val_str = str(val) if val is not None else "???"
        source = a.get("source", "unknown")
        unit = a.get("unit") or ""
        detail = a.get("source_detail") or ""
        print(f"  [{source:<8}] {a['variable']:<25} {val_str + ' ' + unit:<18}")
        print(f"            {detail}")

    questions = result.get("clarification_questions", [])
    if questions:
        print(f"\n--- CLARIFICATION QUESTIONS ({len(questions)}) ---")
        for i, q in enumerate(questions, 1):
            print(f"  {i}. [{q['target_variable']}] {q['question']}")
            print(f"     Why it matters: {q['why_it_matters']}")
    else:
        print("\n  No clarification questions needed.")

print("Helper functions ready.")


Helper functions ready.


---
## Test Case 1 — Clean / Straightforward
Everything is explicitly stated. The LLM should label most things as `stated` with minimal clarification questions.


In [10]:
case_1_input = (
    "I make $2,500 a month after tax and spend about $1,800. "
    "I have $2,000 saved. I want to take a $3,500 trip in 6 months. "
    "Can I afford it?"
)

print("USER INPUT:")
print(f'  "{case_1_input}"')

case_1_result = call_llm(case_1_input)
print_extraction(case_1_result)


USER INPUT:
  "I make $2,500 a month after tax and spend about $1,800. I have $2,000 saved. I want to take a $3,500 trip in 6 months. Can I afford it?"

  Goal: The user wants to know if they can afford a $3,500 trip in 6 months, given their current income, expenses, and savings.
  Confidence: High confidence in extracting all stated financial variables and inferring the monthly surplus.

--- ASSUMPTIONS ---
  [stated  ] monthly_income            2500 $/month      
            User explicitly stated their after-tax monthly income.
  [stated  ] monthly_expenses          1800 $/month      
            User explicitly stated their monthly spending.
  [inferred] monthly_surplus           700 $/month       
            Calculated as monthly_income - monthly_expenses.
  [stated  ] current_savings           2000 $            
            User explicitly stated their current savings.
  [stated  ] goal_amount               3500 $            
            User explicitly stated the cost of their 

## Test Case 2 — Missing Information
Income and expenses are never mentioned. The LLM should flag these as `missing` and ask about them.


In [11]:
case_2_input = (
    "I have around $1,500 saved and want to buy a $2,000 laptop "
    "sometime next semester. Can I afford it?"
)

print("USER INPUT:")
print(f'  "{case_2_input}"')

case_2_result = call_llm(case_2_input)
print_extraction(case_2_result)


USER INPUT:
  "I have around $1,500 saved and want to buy a $2,000 laptop sometime next semester. Can I afford it?"

  Goal: The user wants to know if they can afford a $2,000 laptop by sometime next semester, given their current savings of $1,500.
  Confidence: Confidence is moderate due to missing income, expenses, and a precise deadline for the goal.

--- ASSUMPTIONS ---
  [missing ] monthly_income            ??? $/month       
            User did not state their monthly income.
  [missing ] monthly_expenses          ??? $/month       
            User did not state their monthly expenses.
  [missing ] monthly_surplus           ??? $/month       
            Monthly income and expenses are missing, so surplus cannot be inferred.
  [stated  ] current_savings           1500 $            
            User stated they have around $1,500 saved.
  [stated  ] goal_amount               2000 $            
            User stated the laptop costs $2,000.
  [missing ] goal_deadline           

## Test Case 3 — Messy / Ambiguous
Ranges, vague language, emotional constraints ("don't want to use all of it"). This is the hardest case — check that the LLM handles ranges, breaks out expense categories, and asks smart questions.


In [12]:
case_3_input = (
    "I make around $2k most months from my campus job and tutoring, "
    "but it changes. Rent is like $900 and I probably spend another "
    "$700–800? I have $1,200 saved but don't really want to use all "
    "of it. My friends want to go to Europe next May and it'll "
    "probably be around $3k. Is that realistic?"
)

print("USER INPUT:")
print(f'  "{case_3_input}"')

case_3_result = call_llm(case_3_input)
print_extraction(case_3_result)


USER INPUT:
  "I make around $2k most months from my campus job and tutoring, but it changes. Rent is like $900 and I probably spend another $700–800? I have $1,200 saved but don't really want to use all of it. My friends want to go to Europe next May and it'll probably be around $3k. Is that realistic?"

  Goal: The user wants to know if saving approximately $3,000 for a trip to Europe by next May is realistic, given their current income, expenses, and savings.
  Confidence: High confidence in extracting stated values; some inference made for expense totals and goal deadline based on typical calendar assumptions.

--- ASSUMPTIONS ---
  [stated  ] monthly_income            2000 $/month      
            User stated 'around $2k most months'
  [stated  ] monthly_expenses_rent     900 $/month       
            User stated 'Rent is like $900'
  [stated  ] monthly_expenses_other    750 $/month       
            User stated 'probably spend another $700–800?', midpoint used
  [inferred] mon

---
## Simulated User Correction (Case 2)
This shows the **review-and-fix loop**: the user answers the clarification questions, and the LLM updates the extraction. Watch for:
- `missing` → `stated` label changes
- Clarification questions disappearing once answered


In [13]:
user_correction = (
    "I make about $1,800/month from my part-time job and I spend "
    "around $1,300. I want the laptop by February, so about 4 "
    "months from now."
)

correction_prompt = f"""The user originally said:
\"{case_2_input}\"

You previously extracted these assumptions:
{json.dumps(case_2_result, indent=2)}

The user has now provided corrections and clarifications:
\"{user_correction}\"

Update the assumptions JSON to reflect the user's corrections.
Change the source to "stated" for any variable the user just clarified.
Remove any clarification questions that are now answered.
Return the updated JSON in the same schema as before."""

print("USER CORRECTION:")
print(f'  "{user_correction}"')

case_2_updated = call_llm(correction_prompt)
print_extraction(case_2_updated)

# Show what changed
print("\n--- WHAT CHANGED ---")
orig = {a["variable"]: a["source"] for a in case_2_result.get("assumptions", [])}
updated = {a["variable"]: a["source"] for a in case_2_updated.get("assumptions", [])}
for var, new_src in updated.items():
    old_src = orig.get(var)
    if old_src and old_src != new_src:
        print(f"  {var}: {old_src} → {new_src}")


USER CORRECTION:
  "I make about $1,800/month from my part-time job and I spend around $1,300. I want the laptop by February, so about 4 months from now."

  Goal: The user wants to know if they can afford a $2,000 laptop by February (4 months from now), given their current savings of $1,500, monthly income of $1,800, and monthly expenses of $1,300.
  Confidence: Confidence is high as all critical financial variables and the goal deadline have been stated or inferred.

--- ASSUMPTIONS ---
  [stated  ] monthly_income            1800 $/month      
            User stated they make about $1,800/month from their part-time job.
  [stated  ] monthly_expenses          1300 $/month      
            User stated they spend around $1,300.
  [inferred] monthly_surplus           500 $/month       
            Calculated from stated monthly income ($1,800) minus stated monthly expenses ($1,300).
  [stated  ] current_savings           1500 $            
            User stated they have around $1,50

---
## Raw JSON Output
Copy this and share it with your team — Sivani needs this schema for the calculator, Phoebe needs it for the UI assumption cards.


In [14]:
print("=== Case 3 (messy) — raw JSON for your team ===\n")
print(json.dumps(case_3_result, indent=2))


=== Case 3 (messy) — raw JSON for your team ===

{
  "assumptions": [
    {
      "variable": "monthly_income",
      "value": 2000,
      "unit": "$/month",
      "source": "stated",
      "source_detail": "User stated 'around $2k most months'"
    },
    {
      "variable": "monthly_expenses_rent",
      "value": 900,
      "unit": "$/month",
      "source": "stated",
      "source_detail": "User stated 'Rent is like $900'"
    },
    {
      "variable": "monthly_expenses_other",
      "value": 750,
      "unit": "$/month",
      "source": "stated",
      "source_detail": "User stated 'probably spend another $700\u2013800?', midpoint used"
    },
    {
      "variable": "monthly_expenses",
      "value": 1650,
      "unit": "$/month",
      "source": "inferred",
      "source_detail": "Sum of stated rent and other expenses"
    },
    {
      "variable": "monthly_surplus",
      "value": 350,
      "unit": "$/month",
      "source": "inferred",
      "source_detail": "Calculated from

---
## Try Your Own Input
Paste any messy financial question below to test the extraction.


In [15]:
# ⬇️ CHANGE THIS TO WHATEVER YOU WANT TO TEST ⬇️
my_input = (
    "I just got a raise to 55k but I have 12k in student loans at 6% "
    "interest and my car payment is $350/month. I want to move to a new "
    "apartment that costs $1400/month instead of my current $1000. "
    "Can I swing it?"
)

print("USER INPUT:")
print(f'  "{my_input}"')

my_result = call_llm(my_input)
print_extraction(my_result)


USER INPUT:
  "I just got a raise to 55k but I have 12k in student loans at 6% interest and my car payment is $350/month. I want to move to a new apartment that costs $1400/month instead of my current $1000. Can I swing it?"

  Goal: The user wants to know if they can afford to move into a new apartment with a monthly rent of $1400, given their new salary and existing debts.
  Confidence: Some key expense details are missing, which limits the confidence in a complete financial picture.

--- ASSUMPTIONS ---
  [inferred] monthly_income            4583.33 $/month   
            Annual salary of $55,000 divided by 12 months
  [missing ] monthly_expenses          ??? $/month       
            Total monthly expenses not explicitly stated
  [stated  ] monthly_car_payment       350 $/month       
            User stated car payment is $350/month
  [missing ] monthly_student_loan_payment ??? $/month       
            Student loan payment amount not specified, only total debt and interest rate

In [17]:
# Case 4 — Vague vibes, barely any numbers
case_4_input = "my friends wanna do spring break in miami but idk if i can afford it lol"

print("Case 4 — Vague vibes")
print(f'  "{case_4_input}"')
case_4_result = call_llm(case_4_input)
print_extraction(case_4_result)

Case 4 — Vague vibes
  "my friends wanna do spring break in miami but idk if i can afford it lol"

  Goal: The user wants to know if they can afford a spring break trip to Miami with friends.
  Confidence: Low confidence due to many critical financial variables being missing.

--- ASSUMPTIONS ---
  [missing ] monthly_income            ??? $/month       
            User did not mention their income.
  [missing ] monthly_expenses          ??? $/month       
            User did not mention their expenses.
  [missing ] monthly_surplus           ??? $/month       
            Cannot infer surplus without income and expenses.
  [missing ] current_savings           ??? $             
            User did not mention their current savings.
  [missing ] goal_amount               ??? $             
            User did not specify the cost of the trip.
  [missing ] goal_deadline             ??? months        
            User mentioned 'spring break' but not a specific date or timeframe.
  [st

In [18]:
# Case 5 — Some numbers but wrong framing (annual not monthly, mixed units)
case_5_input = "i make like 24k a year from work study and my parents send me 200 sometimes. rent is 950 with utilities. thinking about getting a car for maybe 8 grand?"

print("Case 5 — Mixed units, irregular income")
print(f'  "{case_5_input}"')
case_5_result = call_llm(case_5_input)
print_extraction(case_5_result)

Case 5 — Mixed units, irregular income
  "i make like 24k a year from work study and my parents send me 200 sometimes. rent is 950 with utilities. thinking about getting a car for maybe 8 grand?"

  Goal: The user wants to know if they can afford to buy a car for approximately $8,000.
  Confidence: Most key variables are stated or reasonably inferred, but additional expenses and current savings are missing.

--- ASSUMPTIONS ---
  [inferred] monthly_income            2000 $/month      
            Annual income of $24,000 divided by 12 months. Parent contribution is too irregular to include.
  [inferred] monthly_expenses          950 $/month       
            Only rent and utilities were mentioned, assuming this is the primary expense.
  [stated  ] monthly_expenses_rent_utilities 950 $/month       
            User stated rent is $950 with utilities.
  [inferred] monthly_surplus           1050 $/month      
            Calculated as monthly_income - monthly_expenses.
  [missing ] curre

In [19]:
# Case 7 — Contradictory / self-correcting mid-sentence
case_7_input = "so i need a new laptop for school, probably like 1500, well actually maybe i could get away with a cheaper one like 800? i have some money saved but i also owe my friend 300. i work at starbucks but my hours are random"

print("Case 7 — Contradictory, self-correcting")
print(f'  "{case_7_input}"')
case_7_result = call_llm(case_7_input)
print_extraction(case_7_result)

Case 7 — Contradictory, self-correcting
  "so i need a new laptop for school, probably like 1500, well actually maybe i could get away with a cheaper one like 800? i have some money saved but i also owe my friend 300. i work at starbucks but my hours are random"

  Goal: The user wants to know if they can afford a new laptop for school, costing between $800 and $1500, while also considering a $300 debt to a friend.
  Confidence: Confidence is moderate due to several key financial variables being missing or given as a range.

--- ASSUMPTIONS ---
  [missing ] monthly_income            ??? $/month       
            User mentioned working but not income amount or frequency
  [missing ] monthly_expenses          ??? $/month       
            User did not mention any regular expenses
  [missing ] monthly_surplus           ??? $/month       
            Cannot infer without income and expenses
  [missing ] current_savings           ??? $             
            User mentioned having 'some 

---
## Switching to AWS Bedrock Later
When your course AWS credits are ready, just change the client setup. The **system prompt and JSON schema stay the same**.


In [16]:
# UNCOMMENT THIS BLOCK WHEN YOU HAVE AWS CREDENTIALS
# Then restart the notebook and run from the top.

# import boto3, json as _json
#
# bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
#
# def call_llm(user_message: str) -> dict:
#     body = _json.dumps({
#         "anthropic_version": "bedrock-2023-05-31",
#         "max_tokens": 2048,
#         "system": SYSTEM_PROMPT,
#         "messages": [{"role": "user", "content": user_message}],
#     })
#     response = bedrock.invoke_model(
#         modelId="us.anthropic.claude-sonnet-4-6",
#         contentType="application/json",
#         accept="application/json",
#         body=body,
#     )
#     result = _json.loads(response["body"].read())
#     raw_text = result["content"][0]["text"]
#     cleaned = raw_text.strip()
#     if cleaned.startswith("```"):
#         cleaned = cleaned.split("\n", 1)[1]
#         cleaned = cleaned.rsplit("```", 1)[0]
#     return _json.loads(cleaned)
#
# print("✅ Switched to Bedrock (Claude)")
